# Movie Expert Agent

An OpenAI function-calling agent that answers movie questions using the [Nomad Movies API](https://nomad-movies-2.nomadcoders.workers.dev).

**Tools**: `get_popular_movies`, `get_movie_details(id)`, `get_movie_credits(id)`. The LLM picks which to call based on the user's question.

In [1]:
import os

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"

In [2]:
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").text


def get_movie_details(id):
    return requests.get(f"{BASE_URL}/movies/{id}").text


def get_movie_credits(id):
    return requests.get(f"{BASE_URL}/movies/{id}/credits").text

In [3]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of currently popular movies. Use this when the user asks what movies are popular, trending, or currently in theaters. Takes no arguments.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a single movie by its ID, such as its title, overview, release date and rating. Use this when the user asks about a specific movie identified by an ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get details for. e.g. 550",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew (actors, director, etc.) of a movie by its ID. Use this when the user asks who starred in, acted in, or made a specific movie identified by an ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get the credits for. e.g. 550",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [5]:
import openai
import json
from openai.types.chat import ChatCompletionMessage

client = openai.OpenAI()

messages = []


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        # 1. Record the assistant's "I want to call a tool" message
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # 2. Actually run each requested function
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with arguments: {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON arguments: {e}")
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            # 3. Feed the tool result back into the conversation
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        # 4. Let the LLM see the result and produce a final answer (recursion)
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [6]:
while True:
    message = input("Ask the Movie Expert... (q to quit) ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 나는 SF 영화를 좋아해
AI: SF 영화는 정말 흥미로운 장르죠! 최근에 인기 있는 SF 영화나 추천을 원하시나요?
User: 인셉션이랑 인터스텔라는 이미 봤어
Calling function: get_popular_movies with arguments: {}
AI: 최근 인기 있는 SF 영화 중 몇 가지를 소개해드릴게요:

1. **Project Hail Mary**
   - ![Project Hail Mary](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)
   - **개요**: 과학 교사인 라이랜드 그레이스는 자신이 누구인지, 어떻게 그곳에 왔는지 기억을 잃은 채 우주선에서 깨어난다. 그의 기억이 돌아오면서 그는 태양을 죽게 만드는 신비로운 물질의 수수께끼를 풀어야 한다.
   - **개봉일**: 2026-03-15
   - **평점**: 8.7

2. **Masters of the Universe**
   - ![Masters of the Universe](https://image.tmdb.org/t/p/w780/3YMd9Ogae4rDKLWuAZFuse9xhc5.jpg)
   - **개요**: 15년 간 떨어져 지낸 후, 파워의 검이 아담 왕자를 에터니아로 인도하면서 그는 스켈레토르의 지배 아래에서 파괴된 자신의 집을 발견한다.
   - **개봉일**: 2026-06-03
   - **평점**: 7.43

3. **The Mandalorian and Grogu**
   - ![The Mandalorian and Grogu](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1HOsiZceDMbRr1Ca.jpg)
   - **개요**: 악명 높은 제국이 무너지고, 제국의 군벌이 은하계를 떠도는 상황에서 새로운 공화국이 반란군이 싸운 모든 것을 보호하기 위해 전설적인 바라파이트 헌터와 그의 어린 제자를 모집하게 된다.
   - **개봉일**: 202